# Voice search

Speech query to a news or product search. Import the pipeline, load the models, check intent and query extraction, then open the Gradio demo.


## Imports

Load the record, pipeline stages, logging, and environment helper.


In [1]:
import os
import subprocess
import sys
from pathlib import Path

if os.getenv("COLAB_RELEASE_TAG"):
    repo = Path("/content/VoiceSearch")
    if not Path("src/app/pipeline.py").is_file():
        if not (repo / "src/app/pipeline.py").is_file():
            subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/ranjanrakesh47g/VoiceSearch.git", str(repo)])
        os.chdir(repo)
    packages = [line.strip() for line in Path("requirements.txt").read_text().splitlines() if line.strip() and not line.startswith("torch")]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

In [2]:
from src.utils.voice_search_model import VoiceSearch
from src.utils.transcribe import Transcriber
from src.utils.intent import IntentDetector
from src.utils.query import QueryExtractor
from src.utils.search import Searcher
from src.utils.logging_setup import setup_logging
from src.app.gradio_demo import build_demo
from src.app.pipeline import VoiceSearchPipeline
from dotenv import load_dotenv

## Environment

Read `.env` and write logs under `logs/`.


In [3]:
load_dotenv()
if os.getenv("COLAB_RELEASE_TAG") and not os.getenv("BRAVE_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["BRAVE_API_KEY"] = userdata.get("BRAVE_API_KEY")
    except Exception:
        pass
setup_logging()

<Logger voice_search (INFO)>

## Models

Load the speech, intent, and named-entity models, then build the searcher.


In [4]:
transcriber = Transcriber()
intent_detector = IntentDetector()
query_extractor = QueryExtractor(intent_detector)
searcher = Searcher(query_extractor)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

asr_freq: 16000


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

## Checks

Run fixed phrases and compare the detected intent and searchable query.


In [5]:
cases = [
    ("Well tell me some news about Tom Cruise.", "news", "Tom Cruise"),
    ("What's up with Trump these days?", "news", "Trump"),
    ("Show me some t-shirts from Amazon.com", "ecommerce", "t-shirts"),
    ("get me some nyke shoes from amazon.com", "ecommerce", "nyke shoes"),
    ("find me some shoes similar to elon musk", "ecommerce", "shoes"),
    ("Tell me the latest headlines about Barack Obama.", "news", "Barack Obama"),
    ("What is happening in New York these days?", "news", "New York"),
    ("Any recent news about NASA?", "news", "NASA"),
    ("Find me wireless headphones from eBay.", "ecommerce", "wireless headphones"),
    ("Show me Adidas running shoes.", "ecommerce", "Adidas running shoes"),
    ("I want a jacket like the one Taylor Swift wore.", "ecommerce", "jacket"),
    ("What's the news about Apple's new iPhone?", "news", "Apple"),
    ("get me some addidas sneakers from flipkart", "ecommerce", "addidas sneakers"),
    ("Trump news", "news", "Trump"),
    ("looking for a cheap office chair", "ecommerce", "office chair"),
]

ok = 0
for text, actual_intent, actual_query in cases:
    record = VoiceSearch(user_query_text=text)
    intent_detector.detect_intent(record)
    query_extractor.extract_query(record)
    intent = record.intent
    query = record.searchable_query
    match = intent == actual_intent and query == actual_query
    ok += match
    print(text)
    print(f"{'ok' if match else 'WRONG'} ----- intent: {intent}, query: {query!r}")
    print()
print(f"ok: {ok}/{len(cases)}")

Well tell me some news about Tom Cruise.
ok ----- intent: news, query: 'Tom Cruise'

What's up with Trump these days?
ok ----- intent: news, query: 'Trump'

Show me some t-shirts from Amazon.com
ok ----- intent: ecommerce, query: 't-shirts'

get me some nyke shoes from amazon.com
ok ----- intent: ecommerce, query: 'nyke shoes'

find me some shoes similar to elon musk
ok ----- intent: ecommerce, query: 'shoes'

Tell me the latest headlines about Barack Obama.
ok ----- intent: news, query: 'Barack Obama'

What is happening in New York these days?
ok ----- intent: news, query: 'New York'

Any recent news about NASA?
ok ----- intent: news, query: 'NASA'

Find me wireless headphones from eBay.
ok ----- intent: ecommerce, query: 'wireless headphones'

Show me Adidas running shoes.
ok ----- intent: ecommerce, query: 'Adidas running shoes'

I want a jacket like the one Taylor Swift wore.
ok ----- intent: ecommerce, query: 'jacket'

What's the news about Apple's new iPhone?
ok ----- intent: new

## Demo

Open the microphone and file-upload UI on the models loaded above.


In [ ]:
demo = build_demo(VoiceSearchPipeline(transcriber, intent_detector, query_extractor, searcher))
if os.getenv("COLAB_RELEASE_TAG"):
    demo.launch(share=True, debug=True)
else:
    demo.launch(share=True, server_port=7680)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://667d710eb20027573e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/components/json_component.py", line 146, in 

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://667d710eb20027573e.gradio.live


## Shut down

Shut down the demo server.


In [ ]:
demo.close()

Closing server running on port: 7680
